# 03 — Forecasting backtest

Walk-forward CV (1-month horizon) over monthly debit aggregates per category.
Compares Naive / Mean3 / SES / ARIMA, reports MAPE & RMSE, plots history + best forecast.

Used as evaluation chapter for the thesis.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
for p in (str(ROOT), str(ROOT / 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from finance.db import get_session
from finance.ml.forecasting.pipeline import (
    build_monthly_series, evaluate_walk_forward, forecast_best,
)
from finance.ml.forecasting.registry import FORECASTERS

print('Forecasters available:', list(FORECASTERS))

## Load real transactions from DB

In [ ]:
from sqlalchemy import select
from finance.domain.models import Transaction

with next(get_session()) as session:
    rows = session.execute(select(Transaction)).scalars().all()
    df = pd.DataFrame([
        {
            'booking_date': r.booking_date,
            'amount': float(r.amount),
            'direction': r.direction,
            'merchant': r.merchant or '',
            'category': r.category or r.category_predicted,
        }
        for r in rows
    ])
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['abs_amount'] = df['amount'].abs()
print(f'Loaded {len(df)} transactions, {df.booking_date.min().date()} to {df.booking_date.max().date()}')
df.head()

## Total monthly debit — overall backtest

In [ ]:
total = build_monthly_series(df, category=None, direction='debit')
print('Months:', len(total))
print(total)

total.plot(marker='o', figsize=(9, 3), title='Monthly debit total (PLN)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
metrics = evaluate_walk_forward(total, horizon=1, min_train=4)
pd.DataFrame(metrics).T.round(2)

In [ ]:
best = forecast_best(total, horizon=3)
print(f'Selected: {best.name}  MAPE={best.mape:.1f}%  RMSE={best.rmse:.0f}')

fig, ax = plt.subplots(figsize=(9, 4))
total.plot(ax=ax, marker='o', label='history')
best.forecast.plot(ax=ax, marker='s', linestyle='--', color='crimson', label=f'forecast ({best.name})')
ax.set_title(f'Total monthly debit — history + 3 mo forecast')
ax.set_ylabel('PLN'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Per-category backtest

In [ ]:
categories = sorted([c for c in df['category'].dropna().unique()])
results = []
for cat in categories:
    s = build_monthly_series(df, category=cat, direction='debit')
    if len(s) < 5:
        continue
    m = evaluate_walk_forward(s, horizon=1, min_train=4)
    for name, vals in m.items():
        results.append({'category': cat, 'model': name, 'months': len(s), **vals})

metrics_df = pd.DataFrame(results)
metrics_df.round(2)

In [ ]:
if not metrics_df.empty:
    pivot = metrics_df.pivot_table(index='category', columns='model', values='rmse').round(0)
    print('RMSE per (category, model):')
    display(pivot)

    best_per_cat = metrics_df.loc[metrics_df.groupby('category')['rmse'].idxmin(),
                                  ['category', 'model', 'mape', 'rmse', 'months']]
    print('\nBest model per category:')
    display(best_per_cat.sort_values('rmse').round(2))

## Conclusions

- Series length is short (≤ 9 months), so ARIMA may degrade or fit poorly; SES + Naive are reliable baselines.
- `forecast_best` is selected by lowest CV-RMSE on each series — falls back to Naive when single-fold history is too short for ML methods.
- Production endpoint `/forecast` reuses exactly this pipeline.